[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module3/03-browser-automation.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module3/03-browser-automation.ipynb)

# Module 3.3 — Browser Automation with Selenium
**Module 3: Automation & Scripting** | Estimated time: 45 minutes

---

## Learning Objectives
By the end of this notebook you will be able to:
- Install and configure Selenium with headless Chrome in Google Colab
- Locate elements using `By.CSS_SELECTOR`, `By.XPATH`, `By.ID`, and `By.CLASS_NAME`
- Use explicit waits (`WebDriverWait` + `expected_conditions`) and implicit waits
- Interact with forms, buttons, and dropdowns programmatically
- Capture screenshots and handle JavaScript alerts
- Build a practical end-to-end form automation example

In [ ]:
# Install Selenium and the Chrome driver manager
!pip install selenium webdriver-manager -q

# Colab already has Chrome installed; make sure we have the matching ChromeDriver
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager
import time
import base64
from pathlib import Path

print('Selenium version:', webdriver.__version__)
print('All imports OK.')

## 1. Colab-Specific Chrome Setup

Google Colab has no display, so Chrome must run in **headless** mode. We also disable the sandbox and GPU to avoid permission errors inside the container.

In [ ]:
def make_driver(headless: bool = True) -> webdriver.Chrome:
    """Create a Chrome WebDriver configured for Colab."""
    options = Options()
    if headless:
        options.add_argument('--headless=new')  # new headless mode (Chrome 112+)
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1280,900')
    options.add_argument(
        '--user-agent=Mozilla/5.0 (X11; Linux x86_64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    )
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.implicitly_wait(5)  # global implicit wait (seconds)
    return driver

driver = make_driver()
print('Driver started. Browser:', driver.capabilities['browserName'],
      driver.capabilities['browserVersion'])

## 2. Navigating and Inspecting a Page

In [ ]:
driver.get('https://books.toscrape.com/')
print('Title   :', driver.title)
print('URL     :', driver.current_url)
print('Page len:', len(driver.page_source), 'chars')

# Implicit wait: driver.find_element will retry for up to 5 s before raising
body = driver.find_element(By.TAG_NAME, 'body')
print('Body tag found:', body.tag_name)

## 3. Finding Elements — All Locator Strategies

| Strategy | Example |
|---|---|
| `By.ID` | `find_element(By.ID, 'username')` |
| `By.NAME` | `find_element(By.NAME, 'email')` |
| `By.CLASS_NAME` | `find_element(By.CLASS_NAME, 'btn-primary')` |
| `By.TAG_NAME` | `find_elements(By.TAG_NAME, 'a')` |
| `By.CSS_SELECTOR` | `find_element(By.CSS_SELECTOR, 'div.card > h2')` |
| `By.XPATH` | `find_element(By.XPATH, '//button[@type="submit"]')` |
| `By.LINK_TEXT` | `find_element(By.LINK_TEXT, 'Next')` |
| `By.PARTIAL_LINK_TEXT` | `find_element(By.PARTIAL_LINK_TEXT, 'Nex')` |

In [ ]:
# CSS selector — all book titles
books = driver.find_elements(By.CSS_SELECTOR, 'article.product_pod h3 a')
print(f'Found {len(books)} books on page 1')
for b in books[:5]:
    print(f'  {b.get_attribute("title")}')
print()

# XPath — find the 'next' button
try:
    next_btn = driver.find_element(By.XPATH, '//li[@class="next"]/a')
    print('Next button text :', next_btn.text)
    print('Next button href :', next_btn.get_attribute('href'))
except NoSuchElementException:
    print('No next button found.')
print()

# By.PARTIAL_LINK_TEXT
try:
    el = driver.find_element(By.PARTIAL_LINK_TEXT, 'Mystery')
    print('Partial link found:', el.text)
except NoSuchElementException:
    print('Partial link not found on this page.')

## 4. Explicit Waits — `WebDriverWait` + `expected_conditions`

Implicit waits apply globally to every `find_element` call. **Explicit waits** let you wait for a specific condition before proceeding, which is essential for JavaScript-rendered content.

Common expected conditions:
- `EC.presence_of_element_located(locator)` — element exists in DOM
- `EC.visibility_of_element_located(locator)` — element is visible
- `EC.element_to_be_clickable(locator)` — element is clickable
- `EC.text_to_be_present_in_element(locator, text)` — text appears in element
- `EC.url_contains(text)` — URL contains string
- `EC.alert_is_present()` — a JS alert is open

In [ ]:
# Navigate to a book detail page
driver.get('https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html')

# Wait up to 10 s for the product main div to appear
try:
    wait = WebDriverWait(driver, timeout=10)
    product_div = wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, 'div.product_main'))
    )
    print('Product div appeared!')
    title = product_div.find_element(By.TAG_NAME, 'h1').text
    price = product_div.find_element(By.CSS_SELECTOR, 'p.price_color').text
    stock = product_div.find_element(By.CSS_SELECTOR, 'p.availability').text.strip()
    print(f'Title : {title}')
    print(f'Price : {price}')
    print(f'Stock : {stock}')
except TimeoutException:
    print('Timed out waiting for product div.')

## 5. Filling Forms and Clicking Buttons

We use the Selenium Playground at `https://the-internet.herokuapp.com/` — a site built specifically for testing automation scripts.

In [ ]:
driver.get('https://the-internet.herokuapp.com/login')
wait = WebDriverWait(driver, 10)

# Wait for the form to load
wait.until(EC.presence_of_element_located((By.ID, 'username')))

# Fill in credentials
username_field = driver.find_element(By.ID, 'username')
password_field = driver.find_element(By.ID, 'password')

username_field.clear()
username_field.send_keys('tomsmith')

password_field.clear()
password_field.send_keys('SuperSecretPassword!')

# Submit the form
login_btn = driver.find_element(By.CSS_SELECTOR, 'button[type="submit"]')
login_btn.click()

# Wait for success message
try:
    flash = wait.until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, 'div.flash.success'))
    )
    print('Login result:', flash.text.strip())
except TimeoutException:
    print('Login may have failed — check credentials.')

print('Current URL:', driver.current_url)

## 6. Handling Dropdowns with `Select`

In [ ]:
driver.get('https://the-internet.herokuapp.com/dropdown')
wait.until(EC.presence_of_element_located((By.ID, 'dropdown')))

dropdown_el = driver.find_element(By.ID, 'dropdown')
select = Select(dropdown_el)

# List all options
print('All options:')
for opt in select.options:
    print(f'  value={opt.get_attribute("value")!r}  text={opt.text!r}')
print()

# Select by visible text
select.select_by_visible_text('Option 2')
print('Selected:', select.first_selected_option.text)

# Select by value attribute
select.select_by_value('1')
print('Selected:', select.first_selected_option.text)

# Select by index (0-based)
select.select_by_index(2)
print('Selected:', select.first_selected_option.text)

## 7. Taking Screenshots

In [ ]:
from IPython.display import Image, display

driver.get('https://books.toscrape.com/')
time.sleep(1)  # let the page settle

screenshot_path = '/tmp/screenshot.png'
driver.save_screenshot(screenshot_path)
print('Screenshot saved to', screenshot_path)
print('File size:', Path(screenshot_path).stat().st_size, 'bytes')

# Display inline in Colab
display(Image(filename=screenshot_path, width=700))

## 8. Handling JavaScript Alerts

In [ ]:
driver.get('https://the-internet.herokuapp.com/javascript_alerts')
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'ul.example li')))

# Trigger a simple alert
driver.find_element(By.XPATH, '//button[text()="Click for JS Alert"]').click()
wait.until(EC.alert_is_present())
alert = driver.switch_to.alert
print('Alert text:', alert.text)
alert.accept()  # click OK

# Trigger a confirm dialog
driver.find_element(By.XPATH, '//button[text()="Click for JS Confirm"]').click()
wait.until(EC.alert_is_present())
confirm = driver.switch_to.alert
print('Confirm text:', confirm.text)
confirm.dismiss()  # click Cancel

# Trigger a prompt dialog
driver.find_element(By.XPATH, '//button[text()="Click for JS Prompt"]').click()
wait.until(EC.alert_is_present())
prompt = driver.switch_to.alert
print('Prompt text:', prompt.text)
prompt.send_keys('Hello, Selenium!')
prompt.accept()

result = driver.find_element(By.ID, 'result').text
print('Result shown on page:', result)

## 9. Practical Example — Automated Form Submission

We will fill out the contact form on `https://the-internet.herokuapp.com/inputs` (number input) as a minimal end-to-end demonstration, then do a full search flow on books.toscrape.com.

In [ ]:
def scrape_search_results(query: str, max_results: int = 5) -> list[dict]:
    """
    Automate a search on books.toscrape.com by navigating the category sidebar.
    Returns a list of {title, price, rating} dicts.
    """
    results = []
    driver.get('https://books.toscrape.com/')
    wait_local = WebDriverWait(driver, 10)

    # Find the category link that contains the query text (case-insensitive)
    links = driver.find_elements(By.CSS_SELECTOR, 'div.side_categories ul li a')
    target = None
    for link in links:
        if query.lower() in link.text.strip().lower():
            target = link
            break

    if target is None:
        print(f'Category "{query}" not found. Available:', [l.text.strip() for l in links[:8]])
        return results

    print(f'Clicking category: {target.text.strip()}')
    target.click()
    wait_local.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'article.product_pod')))

    articles = driver.find_elements(By.CSS_SELECTOR, 'article.product_pod')
    for art in articles[:max_results]:
        title  = art.find_element(By.CSS_SELECTOR, 'h3 a').get_attribute('title')
        price  = art.find_element(By.CSS_SELECTOR, '.price_color').text
        rating = art.find_element(By.CSS_SELECTOR, 'p.star-rating').get_attribute('class').split()[-1]
        results.append({'title': title, 'price': price, 'rating': rating})

    return results

browse_results = scrape_search_results('Mystery', max_results=5)
print(f'\nFound {len(browse_results)} Mystery books:')
for r in browse_results:
    print(f"  {r['rating']:5s} stars  {r['price']}  {r['title'][:50]}")

In [ ]:
# Always quit the driver when done to free resources
driver.quit()
print('Driver closed.')

## Practice Exercises

**Exercise 1 — Page Title Scraper**  
Using Selenium, write a function `get_page_titles(urls: list[str]) -> list[str]` that visits each URL and returns a list of page titles. Use a single driver instance and close it when done.

**Exercise 2 — Checkbox Automation**  
Visit `https://the-internet.herokuapp.com/checkboxes`. There are two checkboxes. Write code that:
1. Reads and prints the current checked state of each checkbox
2. Toggles both checkboxes (checked becomes unchecked and vice versa)
3. Reads and prints the new state to confirm the change

**Exercise 3 — Dynamic Table Reader**  
Visit `https://the-internet.herokuapp.com/tables`. Parse Table 1 into a list of dictionaries where keys are the column headers and values are the cell contents. Print the result as a formatted table.